In [ ]:
# 02 - Feature Extraction

This notebook converts raw PADS smartwatch accelerometer and gyroscope
time-series recordings into interpretable machine-learning features.

Primary analysis:
- Parkinson's disease vs Healthy controls
- 11 standardized movement tasks
- Left and right wrist recordings
- Accelerometer and gyroscope signals
- Participant-level identifiers preserved to prevent data leakage

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import skew, kurtosis
from scipy.signal import periodogram

In [2]:
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data" / "PADS"

PATIENT_DIR = DATA_DIR / "patients"
MOVEMENT_DIR = DATA_DIR / "movement"

print("Dataset found:", DATA_DIR.exists())
print("Patient directory found:", PATIENT_DIR.exists())
print("Movement directory found:", MOVEMENT_DIR.exists())

Dataset found: True
Patient directory found: True
Movement directory found: True


In [5]:
def load_pads_signal(subject_id, task, wrist="LeftWrist"):

    subject_id = str(subject_id).zfill(3)

    observation_path = (
        MOVEMENT_DIR / f"observation_{subject_id}.json"
    )

    with open(observation_path, "r") as f:
        observation = json.load(f)

    session = next(
        session
        for session in observation["session"]
        if session["record_name"] == task
    )

    record = next(
        record
        for record in session["records"]
        if record["device_location"] == wrist
    )

    signal_path = MOVEMENT_DIR / record["file_name"]

    signal_df = pd.read_csv(
        signal_path,
        header=None,
        names=record["channels"]
    )

    return signal_df

In [7]:
test_signal = load_pads_signal(
    "001",
    "TouchNose",
    "LeftWrist"
)

print(test_signal.shape)
display(test_signal.head())

(1024, 7)


,Time,Accelerometer_X,Accelerometer_Y,Accelerometer_Z,Gyroscope_X,Gyroscope_Y,Gyroscope_Z
0,0.000000,-0.314262,-0.406344,-0.647017,1.169301,0.751087,-0.456878
1,0.009687,-0.187177,-0.592680,-0.430099,0.724863,0.728832,-0.282192
2,0.019858,-0.221640,-0.550364,-0.347897,0.449836,0.680792,-0.276376
3,0.029833,-0.190459,-0.423449,-0.322618,0.499378,0.547535,-0.409425
4,0.039866,-0.193140,-0.320662,-0.314953,0.751616,0.513395,-0.755720


In [ ]:
## Feature Extraction

Each smartwatch recording contains six motion channels:
three accelerometer axes and three gyroscope axes.

For each recording, we will calculate interpretable time-domain and
frequency-domain features. We will also calculate acceleration and
gyroscope vector magnitudes to reduce dependence on watch orientation.

In [9]:
def add_signal_magnitudes(df):
    df = df.copy()

    df["Accel_Magnitude"] = np.sqrt(
        df["Accelerometer_X"]**2 +
        df["Accelerometer_Y"]**2 +
        df["Accelerometer_Z"]**2
    )

    df["Gyro_Magnitude"] = np.sqrt(
        df["Gyroscope_X"]**2 +
        df["Gyroscope_Y"]**2 +
        df["Gyroscope_Z"]**2
    )

    return df

In [13]:
test_signal = add_signal_magnitudes(test_signal)

display(test_signal.head())

,Time,Accelerometer_X,Accelerometer_Y,Accelerometer_Z,Gyroscope_X,Gyroscope_Y,Gyroscope_Z,Accel_Magnitude,Gyro_Magnitude
0,0.000000,-0.314262,-0.406344,-0.647017,1.169301,0.751087,-0.456878,0.826139,1.462920
1,0.009687,-0.187177,-0.592680,-0.430099,0.724863,0.728832,-0.282192,0.755838,1.065953
2,0.019858,-0.221640,-0.550364,-0.347897,0.449836,0.680792,-0.276376,0.687792,0.861519
3,0.029833,-0.190459,-0.423449,-0.322618,0.499378,0.547535,-0.409425,0.565391,0.846641
4,0.039866,-0.193140,-0.320662,-0.314953,0.751616,0.513395,-0.755720,0.489206,1.183052


In [15]:
def extract_channel_features(signal, sampling_rate=100):
    signal = np.asarray(signal, dtype=float)

    features = {}

    # Time-domain features
    features["mean"] = np.mean(signal)
    features["std"] = np.std(signal)
    features["median"] = np.median(signal)
    features["min"] = np.min(signal)
    features["max"] = np.max(signal)
    features["range"] = np.ptp(signal)

    features["rms"] = np.sqrt(
        np.mean(signal**2)
    )

    features["iqr"] = (
        np.percentile(signal, 75) -
        np.percentile(signal, 25)
    )

    features["skewness"] = skew(signal)
    features["kurtosis"] = kurtosis(signal)

    # Frequency-domain features
    frequencies, power = periodogram(
        signal,
        fs=sampling_rate
    )

    # Ignore DC component for dominant frequency
    if len(power) > 1:
        dominant_index = np.argmax(power[1:]) + 1
        features["dominant_frequency"] = frequencies[dominant_index]
    else:
        features["dominant_frequency"] = np.nan

    features["spectral_energy"] = np.sum(power)

    return features

In [17]:
example_features = extract_channel_features(
    test_signal["Gyro_Magnitude"],
    sampling_rate=100
)

example_features

{'mean': 3.827620945448028,
 'std': 2.8003241470994693,
 'median': 3.3547541610687857,
 'min': 0.04863481121182,
 'max': 12.597110493409334,
 'range': 12.548475682197514,
 'rms': 4.742625584089558,
 'iqr': 4.817810656460043,
 'skewness': 0.4793344299952682,
 'kurtosis': -0.7530822312249406,
 'dominant_frequency': 1.26953125,
 'spectral_energy': 80.30018896720247}

In [19]:
def extract_recording_features(
    signal_df,
    subject_id,
    task,
    wrist,
    sampling_rate=100
):

    signal_df = add_signal_magnitudes(signal_df)

    feature_row = {
        "subject_id": str(subject_id).zfill(3),
        "task": task,
        "wrist": wrist
    }

    signal_columns = [
        "Accelerometer_X",
        "Accelerometer_Y",
        "Accelerometer_Z",
        "Accel_Magnitude",
        "Gyroscope_X",
        "Gyroscope_Y",
        "Gyroscope_Z",
        "Gyro_Magnitude"
    ]

    for column in signal_columns:

        channel_features = extract_channel_features(
            signal_df[column],
            sampling_rate=sampling_rate
        )

        for feature_name, value in channel_features.items():

            new_column_name = f"{column}_{feature_name}"

            feature_row[new_column_name] = value

    return feature_row

In [21]:
test_features = extract_recording_features(
    signal_df=test_signal,
    subject_id="001",
    task="TouchNose",
    wrist="LeftWrist",
    sampling_rate=100
)

test_features_df = pd.DataFrame([test_features])

print("Shape:", test_features_df.shape)
display(test_features_df)

Shape: (1, 99)


,subject_id,task,wrist,Accelerometer_X_mean,Accelerometer_X_std,Accelerometer_X_median,Accelerometer_X_min,Accelerometer_X_max,Accelerometer_X_range,Accelerometer_X_rms,...,Gyro_Magnitude_median,Gyro_Magnitude_min,Gyro_Magnitude_max,Gyro_Magnitude_range,Gyro_Magnitude_rms,Gyro_Magnitude_iqr,Gyro_Magnitude_skewness,Gyro_Magnitude_kurtosis,Gyro_Magnitude_dominant_frequency,Gyro_Magnitude_spectral_energy
0,001,TouchNose,LeftWrist,-0.186023,0.155923,-0.159208,-0.99051,0.282384,1.272893,0.242727,...,3.354754,0.048635,12.59711,12.548476,4.742626,4.817811,0.479334,-0.753082,1.269531,80.300189


In [23]:
# Check numerical features for missing or infinite values

numeric_test = test_features_df.select_dtypes(include=[np.number])

print("NaN values:", numeric_test.isna().sum().sum())
print("Infinite values:", np.isinf(numeric_test).sum().sum())

NaN values: 0
Infinite values: 0


In [25]:
# Inspect feature ranges

numeric_test.T.describe()

,0
count,96.000000
mean,4.545628
std,16.641408
min,-10.478931
25%,0.061357
50%,0.477340
75%,2.392558
max,111.108202


In [27]:
# Load patient metadata

patient_records = []

for file_path in sorted(PATIENT_DIR.glob("patient_*.json")):
    with open(file_path, "r") as f:
        patient_records.append(json.load(f))

patients_df = pd.DataFrame(patient_records)

binary_cohort = patients_df[
    patients_df["condition"].isin(["Parkinson's", "Healthy"])
].copy()

binary_cohort["subject_id"] = (
    binary_cohort["id"]
    .astype(str)
    .str.zfill(3)
)

binary_cohort["target"] = binary_cohort["condition"].map({
    "Healthy": 0,
    "Parkinson's": 1
})

print(binary_cohort["condition"].value_counts())

condition
Parkinson's    276
Healthy         79
Name: count, dtype: int64


In [29]:
with open(MOVEMENT_DIR / "observation_001.json", "r") as f:
    example_observation = json.load(f)

tasks = [
    session["record_name"]
    for session in example_observation["session"]
]

wrists = ["LeftWrist", "RightWrist"]

print("Tasks:")
print(tasks)

print("\nNumber of tasks:", len(tasks))

Tasks:
['Relaxed', 'RelaxedTask', 'StretchHold', 'LiftHold', 'HoldWeight', 'PointFinger', 'DrinkGlas', 'CrossArms', 'TouchIndex', 'TouchNose', 'Entrainment']

Number of tasks: 11


In [31]:
# Extract features from every recording in the primary cohort

all_feature_rows = []

subject_ids = binary_cohort["subject_id"].tolist()

total_expected = len(subject_ids) * len(tasks) * len(wrists)

print("Participants:", len(subject_ids))
print("Tasks:", len(tasks))
print("Wrists:", len(wrists))
print("Expected recordings:", total_expected)

processed = 0

for subject_id in subject_ids:

    for task in tasks:

        for wrist in wrists:

            signal_df = load_pads_signal(
                subject_id=subject_id,
                task=task,
                wrist=wrist
            )

            feature_row = extract_recording_features(
                signal_df=signal_df,
                subject_id=subject_id,
                task=task,
                wrist=wrist,
                sampling_rate=100
            )

            all_feature_rows.append(feature_row)

            processed += 1

    if processed % 500 < 22:
        print(f"Processed {processed}/{total_expected}")

Participants: 355
Tasks: 11
Wrists: 2
Expected recordings: 7810
Processed 506/7810
Processed 1012/7810
Processed 1518/7810
Processed 2002/7810
Processed 2508/7810
Processed 3014/7810
Processed 3520/7810
Processed 4004/7810
Processed 4510/7810
Processed 5016/7810
Processed 5500/7810
Processed 6006/7810
Processed 6512/7810
Processed 7018/7810
Processed 7502/7810


In [32]:
features_df = pd.DataFrame(all_feature_rows)

print("Feature dataset shape:")
print(features_df.shape)

display(features_df.head())

Feature dataset shape:
(7810, 99)


,subject_id,task,wrist,Accelerometer_X_mean,Accelerometer_X_std,Accelerometer_X_median,Accelerometer_X_min,Accelerometer_X_max,Accelerometer_X_range,Accelerometer_X_rms,...,Gyro_Magnitude_median,Gyro_Magnitude_min,Gyro_Magnitude_max,Gyro_Magnitude_range,Gyro_Magnitude_rms,Gyro_Magnitude_iqr,Gyro_Magnitude_skewness,Gyro_Magnitude_kurtosis,Gyro_Magnitude_dominant_frequency,Gyro_Magnitude_spectral_energy
0,001,Relaxed,LeftWrist,-0.002355,0.004685,-0.002421,-0.023458,0.131153,0.154611,0.005243,...,0.010191,0.000742,0.832345,0.831603,0.027746,0.009336,24.241803,732.420030,0.292969,0.012154
1,001,Relaxed,RightWrist,-0.000341,0.003976,-0.000393,-0.018510,0.126071,0.144581,0.003990,...,0.008726,0.000574,0.888577,0.888003,0.027177,0.007645,26.946439,869.019946,0.244141,0.012470
2,001,RelaxedTask,LeftWrist,-0.001586,0.006423,-0.001676,-0.038445,0.112102,0.150547,0.006616,...,0.032524,0.001250,0.600682,0.599432,0.060239,0.040146,4.204530,36.679898,0.048828,0.033624
3,001,RelaxedTask,RightWrist,0.000329,0.005473,0.000204,-0.022960,0.166025,0.188985,0.005483,...,0.018591,0.001303,0.908047,0.906745,0.035954,0.016057,19.766278,570.043186,0.048828,0.015654
4,001,StretchHold,LeftWrist,-0.001594,0.005821,-0.001792,-0.033946,0.119273,0.153219,0.006035,...,0.020877,0.003180,0.838530,0.835350,0.039254,0.012123,19.827422,463.195996,0.292969,0.010282


In [35]:
label_table = binary_cohort[
    ["subject_id", "condition", "target"]
].copy()

features_labeled_df = features_df.merge(
    label_table,
    on="subject_id",
    how="left",
    validate="many_to_one"
)

print(features_labeled_df.shape)

display(
    features_labeled_df[
        ["subject_id", "task", "wrist", "condition", "target"]
    ].head(20)
)

(7810, 101)


,subject_id,task,wrist,condition,target
0,001,Relaxed,LeftWrist,Healthy,0
1,001,Relaxed,RightWrist,Healthy,0
2,001,RelaxedTask,LeftWrist,Healthy,0
3,001,RelaxedTask,RightWrist,Healthy,0
4,001,StretchHold,LeftWrist,Healthy,0
5,001,StretchHold,RightWrist,Healthy,0
6,001,LiftHold,LeftWrist,Healthy,0
7,001,LiftHold,RightWrist,Healthy,0
8,001,HoldWeight,LeftWrist,Healthy,0
9,001,HoldWeight,RightWrist,Healthy,0


In [37]:
print("Missing labels:")
print(features_labeled_df["condition"].isna().sum())

print("\nTarget distribution by participant:")
print(
    binary_cohort["target"].value_counts()
)

print("\nTarget distribution by recording:")
print(
    features_labeled_df["target"].value_counts()
)

numeric_features = features_labeled_df.select_dtypes(include=[np.number])

print("\nNaN numerical values:")
print(numeric_features.isna().sum().sum())

print("\nInfinite numerical values:")
print(np.isinf(numeric_features).sum().sum())

Missing labels:
0

Target distribution by participant:
target
1    276
0     79
Name: count, dtype: int64

Target distribution by recording:
target
1    6072
0    1738
Name: count, dtype: int64

NaN numerical values:
0

Infinite numerical values:
0


In [39]:
OUTPUT_DIR = PROJECT_DIR / "outputs" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "pads_recording_features.csv"

features_labeled_df.to_csv(
    output_path,
    index=False
)

print("Saved to:")
print(output_path)

Saved to:
C:\Users\Michi\OneDrive\Desktop\CSCI543_PADS_Project\outputs\tables\pads_recording_features.csv


In [41]:
# Final feature dataset QC

feature_columns = [
    col for col in features_labeled_df.columns
    if col not in ["subject_id", "task", "wrist", "condition", "target"]
]

X_check = features_labeled_df[feature_columns]

print("Number of sensor features:", len(feature_columns))
print("NaN values:", X_check.isna().sum().sum())
print("Infinite values:", np.isinf(X_check).sum().sum())

print("\nRows per participant:")
print(features_labeled_df.groupby("subject_id").size().describe())

print("\nUnique participants:")
print(features_labeled_df["subject_id"].nunique())

Number of sensor features: 96
NaN values: 0
Infinite values: 0

Rows per participant:
count    355.0
mean      22.0
std        0.0
min       22.0
25%       22.0
50%       22.0
75%       22.0
max       22.0
dtype: float64

Unique participants:
355
